# 07 — Data Validation

## 1. Objective and validation boundary

Validate the latest successful immutable Step 5 extraction. This notebook detects and reports issues only; it does not clean, impute, standardise, deduplicate, label, split, or model data.

In [1]:
from pathlib import Path
import sys
from IPython.display import Markdown, display
import pandas as pd

PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'src' / 'urban_ops').is_dir())
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from urban_ops.validation.pipeline import run_validation
from urban_ops.validation.severity import severity_counts, validation_exit_code

CONFIG_PATH = PROJECT_ROOT / 'configs/data/validation_rules.yaml'
result = run_validation(config_path=CONFIG_PATH)
tables = result.tables
pd.set_option('display.max_columns', 80)

## 2. Latest successful raw extraction

In [2]:
display(pd.DataFrame([{'run_id': result.metadata.run_id, 'raw_file': str(result.raw_file), 'rows': result.row_count, 'columns': result.column_count, 'raw_sha256': result.raw_sha256_before}]).T)

,0
run_id,20260731T122433Z_7e3a488efc738a9c
raw_file,/Users/mohammadmubashir/VCode/urban-operations...
rows,40017
columns,18
raw_sha256,f4118fe61c953228e6894f0f203e3b2025b7c1ccb88496...


## 3. Extraction metadata reconciliation

In [3]:
display(tables['validation_checks.csv'].query("area == 'metadata'"))

,check_id,area,check_name,severity,status,observed_value,expected_value,affected_rows,affected_rate,message,recommended_step_7_action
2,metadata.dataset_id,metadata,Expected dataset ID,CRITICAL,PASS,erm2-nwe9,erm2-nwe9,0,0.0,Raw provenance must identify the configured NY...,Reject the raw run and re-ingest from the expe...
3,metadata_scope.agency,metadata,Metadata scope agency,CRITICAL,PASS,DSNY,DSNY,0,0.0,Raw-run metadata must agree with the current s...,Stop downstream work and create a new governed...
4,metadata_scope.authority_path,metadata,Metadata scope authority_path,CRITICAL,PASS,/Users/mohammadmubashir/VCode/urban-operations...,/Users/mohammadmubashir/VCode/urban-operations...,0,0.0,Raw-run metadata must agree with the current s...,Stop downstream work and create a new governed...
5,metadata_scope.complaint_type,metadata,Metadata scope complaint_type,CRITICAL,PASS,Graffiti,Graffiti,0,0.0,Raw-run metadata must agree with the current s...,Stop downstream work and create a new governed...
6,metadata_scope.end_date,metadata,Metadata scope end_date,CRITICAL,PASS,2025-12-31,2025-12-31,0,0.0,Raw-run metadata must agree with the current s...,Stop downstream work and create a new governed...
7,metadata_scope.start_date,metadata,Metadata scope start_date,CRITICAL,PASS,2024-01-01,2024-01-01,0,0.0,Raw-run metadata must agree with the current s...,Stop downstream work and create a new governed...


## 4. Authoritative scope reconciliation

In [4]:
display(tables['scope_validation.csv'])

,scope_rule,affected_row_count,affected_row_rate,expected_value,status,severity
0,out_of_scope_agency,0,0.0,DSNY,PASS,CRITICAL
1,out_of_scope_complaint_type,0,0.0,Graffiti,PASS,CRITICAL
2,before_start,0,0.0,2024-01-01,PASS,CRITICAL
3,after_end,0,0.0,2025-12-31,PASS,CRITICAL
4,unparseable_created_date,0,0.0,0,PASS,CRITICAL


## 5. Dataset shape and column inventory

In [5]:
display(tables['column_profile.csv'][['column_name', 'raw_dtype', 'non_null_count', 'null_count', 'distinct_count']])

,column_name,raw_dtype,non_null_count,null_count,distinct_count
0,unique_key,str,40017,0,40017
1,created_date,str,40017,0,39968
2,closed_date,str,39747,270,1769
3,due_date,str,36236,3781,23684
4,agency,str,40017,0,1
5,agency_name,str,40017,0,1
6,complaint_type,str,40017,0,1
7,descriptor,str,40017,0,1
8,descriptor_2,float64,0,40017,0
9,status,str,40017,0,3


## 6. Schema validation

In [6]:
display(tables['schema_validation.csv'])

,column_name,position,raw_dtype,required,selected_in_metadata,semantic_type,status
0,unique_key,0,str,True,True,identifier_string,PASS
1,created_date,1,str,True,True,timestamp,PASS
2,closed_date,2,str,True,True,timestamp,PASS
3,due_date,3,str,True,True,timestamp,PASS
4,agency,4,str,True,True,string_or_nullable_source_value,PASS
5,agency_name,5,str,True,True,string_or_nullable_source_value,PASS
6,complaint_type,6,str,True,True,string_or_nullable_source_value,PASS
7,descriptor,7,str,True,True,string_or_nullable_source_value,PASS
8,descriptor_2,8,float64,True,True,string_or_nullable_source_value,PASS
9,status,9,str,True,True,string_or_nullable_source_value,PASS


## 7. Missingness analysis

In [7]:
display(tables['missingness_summary.csv'].sort_values('null_count', ascending=False))

,column_name,column_role,row_count,non_null_count,null_count,null_rate,empty_string_count,whitespace_only_count,null_like_string_count,missingness_severity,recommended_step_7_action
8,descriptor_2,CONDITIONAL_FEATURE,40017,0,40017,1.000000,0,0,0,WARNING,Preserve nulls until an explicit Step 7 field ...
16,resolution_description,POST_CREATION_FIELD,40017,23188,16829,0.420546,0,0,0,WARNING,Preserve nulls until an explicit Step 7 field ...
14,location_type,CONDITIONAL_FEATURE,40017,33648,6369,0.159157,0,0,0,WARNING,Preserve nulls until an explicit Step 7 field ...
3,due_date,TARGET_INPUT,40017,36236,3781,0.094485,0,0,0,ERROR,Preserve rows and mark target-ineligible; do n...
12,latitude,CONDITIONAL_FEATURE,40017,38616,1401,0.035010,0,0,0,WARNING,Preserve complaint; handle the geographic feat...
13,longitude,CONDITIONAL_FEATURE,40017,38616,1401,0.035010,0,0,0,WARNING,Preserve complaint; handle the geographic feat...
11,incident_zip,CONDITIONAL_FEATURE,40017,38619,1398,0.034935,0,0,0,WARNING,Preserve complaint; handle the geographic feat...
2,closed_date,TARGET_INPUT,40017,39747,270,0.006747,0,0,0,WARNING,Preserve rows and mark target-ineligible; do n...
1,created_date,SAFE_FEATURE,40017,40017,0,0.000000,0,0,0,INFO,No missing-value action required.
0,unique_key,IDENTIFIER,40017,40017,0,0.000000,0,0,0,INFO,No missing-value action required.


## 8. Timestamp parsing analysis

In [8]:
display(tables['timestamp_validation_summary.csv'])

,column_name,non_null_count,parse_success_count,parse_failure_count,parse_success_rate,timezone_aware_count,timezone_naive_count,minimum_timestamp,maximum_timestamp,invalid_value_examples
0,created_date,40017,40017,0,1.0,0,40017,2024-01-01T03:03:35+00:00,2025-12-31T20:39:09+00:00,
1,closed_date,39747,39747,0,1.0,0,39747,2024-01-02T02:22:23+00:00,2026-07-28T00:00:00+00:00,
2,due_date,36236,36236,0,1.0,0,36236,2024-01-09T07:13:00+00:00,2026-07-05T08:51:40+00:00,
3,resolution_action_updated_date,40017,40017,0,1.0,0,40017,2024-01-02T02:22:23+00:00,2026-07-29T06:46:41+00:00,


## 9. Chronology validation

In [9]:
display(tables['chronology_violations.csv'])

,unique_key,created_date_raw,due_date_raw,closed_date_raw,resolution_action_updated_date_raw,violation_type,severity,recommended_step_7_action
0,65549394,2025-07-13T13:22:40.000,2025-08-12T13:22:40.000,2025-07-11T00:00:00.000,2025-07-29T11:03:31.000,closed_before_created,ERROR,Retain as excluded/quarantine evidence; do not...
1,65551559,2025-07-13T13:24:35.000,2025-08-12T13:24:35.000,2025-07-11T00:00:00.000,2025-07-29T11:03:31.000,closed_before_created,ERROR,Retain as excluded/quarantine evidence; do not...
2,65552645,2025-07-13T13:27:00.000,2025-08-12T13:27:00.000,2025-07-11T00:00:00.000,2025-07-29T11:03:31.000,closed_before_created,ERROR,Retain as excluded/quarantine evidence; do not...
3,65549393,2025-07-13T13:29:33.000,2025-08-12T13:29:33.000,2025-07-11T00:00:00.000,2025-07-29T11:03:31.000,closed_before_created,ERROR,Retain as excluded/quarantine evidence; do not...
4,65545090,2025-07-13T13:30:07.000,2025-08-12T13:30:07.000,2025-07-11T00:00:00.000,2025-07-29T11:03:31.000,closed_before_created,ERROR,Retain as excluded/quarantine evidence; do not...
5,65545089,2025-07-13T13:31:16.000,2025-08-12T13:31:16.000,2025-07-11T00:00:00.000,2025-07-29T11:03:31.000,closed_before_created,ERROR,Retain as excluded/quarantine evidence; do not...


## 10. Duplicate analysis

In [10]:
display(tables['duplicate_summary.csv'])

,metric,row_count
0,duplicate_full_rows,0
1,duplicate_unique_key_groups,0
2,rows_in_duplicate_key_groups,0
3,redundant_exact_duplicate_rows,0
4,conflicting_duplicate_keys,0
5,rows_in_conflicting_groups,0
6,missing_unique_key_rows,0


## 11. Category profiling

In [11]:
display(tables['category_profile.csv'].head(50)); display(tables['category_variants.csv'])

,column_name,raw_value,normalised_comparison_value,row_count,row_share,has_leading_whitespace,has_trailing_whitespace,is_empty,is_whitespace_only,is_null_like_string,case_variant_group,is_rare_category,column_distinct_count,is_high_cardinality
0,agency,DSNY,dsny,40017,1.000000,False,False,False,False,False,dsny,False,1,False
1,agency_name,Department of Sanitation,department of sanitation,40017,1.000000,False,False,False,False,False,department of sanitation,False,1,False
2,complaint_type,Graffiti,graffiti,40017,1.000000,False,False,False,False,False,graffiti,False,1,False
3,descriptor,Graffiti,graffiti,40017,1.000000,False,False,False,False,False,graffiti,False,1,False
4,descriptor_2,<NULL>,<NULL>,40017,1.000000,False,False,False,False,False,<NULL>,False,0,False
5,status,Closed,closed,39740,0.993078,False,False,False,False,False,closed,False,3,False
6,status,Open,open,271,0.006772,False,False,False,False,False,open,False,3,False
7,status,Pending,pending,6,0.000150,False,False,False,False,False,pending,True,3,False
8,borough,BROOKLYN,brooklyn,20758,0.518730,False,False,False,False,False,brooklyn,False,6,False
9,borough,MANHATTAN,manhattan,9583,0.239473,False,False,False,False,False,manhattan,False,6,False


,column_name,normalised_comparison_value,raw_variant_count,raw_variants,row_count,variant_type


## 12. Empty-string and null-like-value analysis

In [12]:
display(tables['missingness_summary.csv'][['column_name', 'empty_string_count', 'whitespace_only_count', 'null_like_string_count']])

,column_name,empty_string_count,whitespace_only_count,null_like_string_count
0,unique_key,0,0,0
1,created_date,0,0,0
2,closed_date,0,0,0
3,due_date,0,0,0
4,agency,0,0,0
5,agency_name,0,0,0
6,complaint_type,0,0,0
7,descriptor,0,0,0
8,descriptor_2,0,0,0
9,status,0,0,0


## 13. Geographic validation

In [13]:
display(tables['geographic_validation.csv']); display(tables['geographic_outliers.csv'].head(50))

,field,metric,row_count,row_rate,severity
0,borough,missing,0,0.000000,WARNING
1,borough,unspecified_or_null_like,21,0.000525,WARNING
2,latitude,missing,1401,0.035010,WARNING
3,latitude,parseable,38616,0.964990,INFO
4,latitude,invalid_numeric,0,0.000000,ERROR
5,latitude,outside_world_range,0,0.000000,ERROR
6,longitude,missing,1401,0.035010,WARNING
7,longitude,parseable,38616,0.964990,INFO
8,longitude,invalid_numeric,0,0.000000,ERROR
9,longitude,outside_world_range,0,0.000000,ERROR


,unique_key,field,raw_value,issue_type,severity
0,61281704,borough,Unspecified,unspecified_or_null_like,WARNING
1,61432477,borough,Unspecified,unspecified_or_null_like,WARNING
2,61432476,borough,Unspecified,unspecified_or_null_like,WARNING
3,61588434,borough,Unspecified,unspecified_or_null_like,WARNING
4,62478598,borough,Unspecified,unspecified_or_null_like,WARNING
5,62776695,borough,Unspecified,unspecified_or_null_like,WARNING
6,63586750,borough,Unspecified,unspecified_or_null_like,WARNING
7,64063357,borough,Unspecified,unspecified_or_null_like,WARNING
8,64617882,borough,Unspecified,unspecified_or_null_like,WARNING
9,64748223,borough,Unspecified,unspecified_or_null_like,WARNING


## 14. Target-readiness analysis

In [14]:
display(tables['target_readiness_summary.csv']); display(tables['candidate_exclusion_reason_summary.csv'])

,readiness_rule,true_count,false_count,pass_count,fail_count,pass_rate,provisional
0,has_created_date,40017,0,40017,0,1.000000,True
1,has_due_date,36236,3781,36236,3781,0.905515,True
2,has_closed_date,39747,270,39747,270,0.993253,True
3,valid_due_chronology,40017,0,40017,0,1.000000,True
4,valid_closed_chronology,40011,6,40011,6,0.999850,True
5,status_allowed,39740,277,39740,277,0.993078,True
6,outcome_mature,36236,3781,36236,3781,0.905515,True
7,is_conflicting_duplicate,0,40017,40017,0,1.000000,True
8,candidate_target_eligible,35960,4057,35960,4057,0.898618,True


,candidate_exclusion_reason,row_count,row_share,final_target_construction_stage
0,eligible,35960,0.898618,Step 7 after approved cleaning
1,missing_due_date,3781,0.094485,Step 7 after approved cleaning
2,missing_closed_date,269,0.006722,Step 7 after approved cleaning
3,closed_before_created,6,0.00015,Step 7 after approved cleaning
4,excluded_status,1,0.000025,Step 7 after approved cleaning


## 15. Validation severity summary

In [15]:
display(pd.Series(severity_counts(result.checks), name='finding_count').to_frame()); display(tables['validation_checks.csv'].query("status != 'PASS'"))

,finding_count
CRITICAL,0
ERROR,2
WARNING,12
INFO,0


,check_id,area,check_name,severity,status,observed_value,expected_value,affected_rows,affected_rate,message,recommended_step_7_action
19,chronology.closed_before_created,chronology,Closed Before Created,ERROR,FAIL,6,0,6,0.000150,Timestamp order conflicts with the governed ch...,Retain as excluded/quarantine evidence; do not...
26,missingness.due_date,missingness,Missing due_date,ERROR,FAIL,3781,0,3781,0.094485,Raw nulls in due_date were measured and preser...,Preserve rows and mark target-ineligible; do n...
37,geography.borough.unspecified_or_null_like,geography,borough unspecified_or_null_like,WARNING,WARN,21,0,21,0.000525,Geographic quality was measured without correc...,Preserve complaint and apply an approved featu...
40,geography.incident_zip.missing,geography,incident_zip missing,WARNING,WARN,1398,0,1398,0.034935,Geographic quality was measured without correc...,Preserve complaint and apply an approved featu...
43,geography.latitude.missing,geography,latitude missing,WARNING,WARN,1401,0,1401,0.035010,Geographic quality was measured without correc...,Preserve complaint and apply an approved featu...
44,geography.longitude.missing,geography,longitude missing,WARNING,WARN,1401,0,1401,0.035010,Geographic quality was measured without correc...,Preserve complaint and apply an approved featu...
45,missingness.closed_date,missingness,Missing closed_date,WARNING,WARN,270,0,270,0.006747,Raw nulls in closed_date were measured and pre...,Preserve rows and mark target-ineligible; do n...
46,missingness.descriptor_2,missingness,Missing descriptor_2,WARNING,WARN,40017,0,40017,1.000000,Raw nulls in descriptor_2 were measured and pr...,Preserve nulls until an explicit Step 7 field ...
47,missingness.incident_zip,missingness,Missing incident_zip,WARNING,WARN,1398,0,1398,0.034935,Raw nulls in incident_zip were measured and pr...,Preserve complaint; handle the geographic feat...
48,missingness.latitude,missingness,Missing latitude,WARNING,WARN,1401,0,1401,0.035010,Raw nulls in latitude were measured and preser...,Preserve complaint; handle the geographic feat...


## 16. Proposed Step 7 cleaning actions

In [16]:
display(tables['proposed_cleaning_actions.csv'])

,issue_id,validation_area,source_column,issue_description,affected_rows,severity,proposed_action,action_type,requires_governance_approval,source_rule,notes
0,missingness.closed_date,missingness,closed_date,Raw nulls in closed_date were measured and pre...,270,WARNING,Preserve rows and mark target-ineligible; do n...,STEP_7_RULE,False,Missing closed_date,Recommendation only; Step 6 did not modify any...
1,missingness.due_date,missingness,due_date,Raw nulls in due_date were measured and preser...,3781,ERROR,Preserve rows and mark target-ineligible; do n...,STEP_7_RULE,False,Missing due_date,Recommendation only; Step 6 did not modify any...
2,missingness.descriptor_2,missingness,descriptor_2,Raw nulls in descriptor_2 were measured and pr...,40017,WARNING,Preserve nulls until an explicit Step 7 field ...,STEP_7_RULE,False,Missing descriptor_2,Recommendation only; Step 6 did not modify any...
3,missingness.incident_zip,missingness,incident_zip,Raw nulls in incident_zip were measured and pr...,1398,WARNING,Preserve complaint; handle the geographic feat...,STEP_7_RULE,False,Missing incident_zip,Recommendation only; Step 6 did not modify any...
4,missingness.latitude,missingness,latitude,Raw nulls in latitude were measured and preser...,1401,WARNING,Preserve complaint; handle the geographic feat...,STEP_7_RULE,False,Missing latitude,Recommendation only; Step 6 did not modify any...
5,missingness.longitude,missingness,longitude,Raw nulls in longitude were measured and prese...,1401,WARNING,Preserve complaint; handle the geographic feat...,STEP_7_RULE,False,Missing longitude,Recommendation only; Step 6 did not modify any...
6,missingness.location_type,missingness,location_type,Raw nulls in location_type were measured and p...,6369,WARNING,Preserve nulls until an explicit Step 7 field ...,STEP_7_RULE,False,Missing location_type,Recommendation only; Step 6 did not modify any...
7,null_like.open_data_channel_type,category,open_data_channel_type,String sentinels remain distinct from true nul...,40017,WARNING,Review and map only approved null-equivalent v...,GOVERNANCE_REVIEW,True,Blank or null-like open_data_channel_type,Recommendation only; Step 6 did not modify any...
8,missingness.resolution_description,missingness,resolution_description,Raw nulls in resolution_description were measu...,16829,WARNING,Preserve nulls until an explicit Step 7 field ...,STEP_7_RULE,False,Missing resolution_description,Recommendation only; Step 6 did not modify any...
9,chronology.closed_before_created,chronology,,Timestamp order conflicts with the governed ch...,6,ERROR,Retain as excluded/quarantine evidence; do not...,STEP_7_RULE,False,Closed Before Created,Recommendation only; Step 6 did not modify any...


## 17. Known limitations

The raw API snapshot can reflect later source corrections. Comparison normalisation does not prove semantic equivalence. The broad NYC bounding box is a review signal, not a borough/ZIP correction authority. Candidate target readiness is provisional; final cleaning and target construction belong to Step 7.

## 18. Step 6 completion decision

In [17]:
default_exit_code = validation_exit_code(result.checks, fail_on_error=False, fail_on_warning=False)
decision = 'PASSED: safe to proceed to governed Step 7 cleaning' if default_exit_code == 0 else 'FAILED: critical raw validation issue'
display(Markdown(f'**{decision}.** Overall evidence status: `{result.overall_status}`. Raw file modified: `{result.raw_file_modified}`.'))

**PASSED: safe to proceed to governed Step 7 cleaning.** Overall evidence status: `ERROR`. Raw file modified: `False`.